## Bronze — Estimativas de População Residente (IBGE/SIDRA 6579)

**Fonte:** SIDRA tabela 6579 — [Estimativas de População](https://sidra.ibge.gov.br/pesquisa/estimapop/tabelas)

- Licença: dados públicos IBGE, reutilização autorizada com citação da fonte.
- Área responsável: IBGE — Diretoria de Pesquisas (coordenação de População e Indicadores Sociais).
- **Arquivo origem:** `/Volumes/workspace/raw/IBGE/estimativa_populacional_6579.csv` (UTF-8 com BOM), anos disponíveis **2001 a 2025**, nível município.
- **Formato do arquivo:** 1ª linha = título (`"Tabela 6579 - População residente estimada"`), 2ª linha = cabeçalho (`Cód., Município, Ano, Variável, ""`) — ambas descartadas na leitura; as linhas seguintes são os dados.
- **Grão:** 1 linha por município x ano.
- **Linhagem:** download SIDRA → CSV no Volume → `workspace.bronze.estimativa_populacional`.

In [0]:
%run ./_setup_pop

In [0]:
from pyspark.sql import functions as F
from data_pipeline import read_csv, normalizar_colunas, save_table, add_column_comments
from metadata.metadata import BRONZE_ESTIMATIVA_POPULACAO_COMMENTS

In [0]:
FILE_PATH = "/Volumes/workspace/raw/IBGE/estimativa_populacional_6579.csv"
TABLE_NAME = "workspace.bronze.estimativa_populacional"
CSV_ENCODING = "UTF-8"
CSV_DELIMITER = ","

# Nomes normalizados das colunas do export SIDRA (header original:
# Cód., Município, Ano, Variável e última coluna sem nome)
COLUNAS_ORIGINAIS = ["codigo_municipio", "municipio", "ano", "variavel", "valor"]

In [0]:
# Header=False: a 1ª linha do arquivo é título da tabela e a 2ª é o cabeçalho;
# nomes aplicados manualmente e linhas de controle descartadas abaixo
df_raw = read_csv(
    spark,
    FILE_PATH,
    delimiter=CSV_DELIMITER,
    encoding=CSV_ENCODING,
    header=False,
    infer_schema=False,
)
df_raw = df_raw.toDF(*COLUNAS_ORIGINAIS)
total_bruto = df_raw.count()
print(f"Linhas brutas (inclui título e cabeçalho): {total_bruto:,}")
display(df_raw.limit(5))

# Mantém apenas linhas de dados: 'ano' com 4 dígitos remove título/cabeçalho
df = df_raw.filter(F.col("ano").rlike("^[0-9]{4}$"))
df = normalizar_colunas(df)
print(f"Linhas de dados: {df.count():,} | Removidas (título/cabeçalho): {total_bruto - df.count():,}")
display(df.limit(10))

In [0]:
save_table(df, TABLE_NAME)
add_column_comments(
    spark,
    TABLE_NAME,
    BRONZE_ESTIMATIVA_POPULACAO_COMMENTS
)

In [0]:
total = spark.table(TABLE_NAME).count()
distintos = spark.table(TABLE_NAME).distinct().count()
print(f"Total: {total:,} | Linhas únicas: {distintos:,} | Duplicatas: {total - distintos:,}")
display(spark.sql(f"SELECT ano, count(*) AS qtd_municipios FROM {TABLE_NAME} GROUP BY ano ORDER BY ano"))
display(spark.sql(f"SELECT variavel, count(*) AS qtd FROM {TABLE_NAME} GROUP BY variavel"))